# AI System Design — Estimation Calculators

Interactive back-of-the-envelope calculators for the Chapter 5 formulas: **DAU→QPS, GPU memory, KV cache, embedding storage, and training cost**.

Pure Python — no dependencies. Runs in Jupyter or Google Colab. Edit the numbers in each example cell and re-run.

> Estimates use decimal units (1 GB = 1e9 bytes) to match the book's rounding. Answers are order-of-magnitude — the method matters more than precision.

In [ ]:
# Shared constants
BYTES_PER_PARAM = {"fp32": 4, "fp16": 2, "bf16": 2, "fp8": 1, "int8": 1, "int4": 0.5}
SECONDS_PER_DAY = 86_400

def human_bytes(n):
    for unit in ["B", "KB", "MB", "GB", "TB", "PB"]:
        if abs(n) < 1000:
            return f"{n:.2f} {unit}"
        n /= 1000
    return f"{n:.2f} EB"

## 1. DAU → QPS
`QPS = DAU × requests/user/day ÷ 86,400`, then apply a peak multiplier (2–3× consumer, 5–10× e-commerce spikes, 10–20× news).

In [ ]:
def qps_estimate(dau, requests_per_user_per_day, peak_multiplier=3, active_window_hours=24):
    seconds = active_window_hours * 3600
    daily = dau * requests_per_user_per_day
    avg = daily / seconds
    peak = avg * peak_multiplier
    print(f"Daily requests : {daily:,.0f}")
    print(f"Average QPS    : {avg:,.0f}")
    print(f"Peak QPS ({peak_multiplier}x) : {peak:,.0f}")
    return avg, peak

# Example: 800M DAU video platform, 8 views/user/day, 3x peak
qps_estimate(dau=800_000_000, requests_per_user_per_day=8, peak_multiplier=3);

## 2. GPU memory to serve a model
`memory ≈ params × bytes/param × overhead` (1.5× typical for inference: KV cache + activations).

In [ ]:
def gpu_memory(params_billions, precision="fp16", overhead=1.5, gpu_gb=80):
    bytes_pp = BYTES_PER_PARAM[precision]
    raw = params_billions * 1e9 * bytes_pp
    total = raw * overhead
    gpus = -(-total // (gpu_gb * 1e9))  # ceil
    print(f"Weights ({precision}) : {human_bytes(raw)}")
    print(f"+ {overhead}x overhead : {human_bytes(total)}")
    print(f"GPUs needed (@{gpu_gb} GB) : {int(gpus)}")
    return total

# Example: serve a 70B model in FP16
gpu_memory(params_billions=70, precision="fp16");

## 3. KV cache memory
`KV/token (fp16) ≈ 8 × layers × hidden` bytes. Total = per-token × context length × batch size. KV cache usually limits batch size, not the weights.

In [ ]:
def kv_cache(layers, hidden, context_tokens, batch_size=1, bytes_fp=2):
    # 4 x layers x hidden x bytes_per_value ; = 8*L*H for fp16 (bytes_fp=2)
    per_token = 4 * layers * hidden * bytes_fp
    per_request = per_token * context_tokens
    total = per_request * batch_size
    print(f"KV per token   : {human_bytes(per_token)}")
    print(f"Per request ({context_tokens} tok) : {human_bytes(per_request)}")
    print(f"Batch {batch_size}        : {human_bytes(total)}")
    return total

# Example: Llama-70B (80 layers, hidden 8192), 8K context, batch 32
kv_cache(layers=80, hidden=8192, context_tokens=8000, batch_size=32);

## 4. Embedding / vector-DB storage
`storage = dim × count × bytes/dim`. Add ~1.5× for an HNSW index. Quantizing (fp32→int8) cuts storage 4×.

In [ ]:
def embedding_storage(count, dim, precision="fp32", index_multiplier=1.5, shard_gb=None):
    bytes_pd = BYTES_PER_PARAM[precision]
    raw = count * dim * bytes_pd
    indexed = raw * index_multiplier
    print(f"Raw vectors    : {human_bytes(raw)}")
    print(f"+ index ({index_multiplier}x) : {human_bytes(indexed)}")
    if shard_gb:
        shards = -(-indexed // (shard_gb * 1e9))
        print(f"Shards (@{shard_gb} GB) : {int(shards)}")
    return indexed

# Example: 500M docs, 1536-dim, float32
embedding_storage(count=500_000_000, dim=1536, precision="fp32", shard_gb=400);

## 5. Training / fine-tuning cost
`FLOPs ≈ 6 × params × tokens × epochs`; `GPU-hours = FLOPs ÷ (effective TFLOPS × 1e12 × 3600)`; `cost = GPU-hours × $/GPU-hr × #GPUs`. Note: even with LoRA, the forward pass runs the full model.

In [ ]:
def training_cost(params_billions, tokens, epochs=1, eff_tflops=400, num_gpus=1, cost_per_gpu_hr=2.0):
    flops = 6 * params_billions * 1e9 * tokens * epochs
    gpu_hours = flops / (eff_tflops * 1e12 * 3600)
    wall_clock = gpu_hours / num_gpus
    cost = gpu_hours * cost_per_gpu_hr
    print(f"FLOPs        : {flops:.2e}")
    print(f"GPU-hours    : {gpu_hours:,.1f}")
    print(f"Wall-clock   : {wall_clock:,.1f} h on {num_gpus} GPUs")
    print(f"Cost         : ${cost:,.0f}")
    return gpu_hours, cost

# Example: pretrain 7B on 1T tokens, 512 H100s @400 eff TFLOPS, $2/GPU-hr
training_cost(params_billions=7, tokens=1e12, epochs=1, eff_tflops=400, num_gpus=512, cost_per_gpu_hr=2.0)
print("---")
# Example: LoRA fine-tune 13B, 5M tokens, 3 epochs, 1 A100 @300 eff TFLOPS, $3.50/hr
training_cost(params_billions=13, tokens=5e6, epochs=3, eff_tflops=300, num_gpus=1, cost_per_gpu_hr=3.5);

## Try your own
Copy any function call above into the cell below and change the inputs to match your interview scenario.

In [ ]:
# your scenario here
